# What must be known

The Rietveld fit takes the pulse pattern, the open beam and the edge shape
(the nbragg response) as known. This notebook estimates how errors in these
inputs bias the result, and what freeing them costs in precision.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import nbragg
import frame_overlap as fo

# the material, as in nbragg: 2 cm of alpha-iron
xs = nbragg.CrossSection(materials={"iron": {"mat": "Fe_sg229_Iron-alpha.ncmat"}})
model = nbragg.TransmissionModel(xs, response="jorgensen_inv", tof_length=9.0, vary_response=False)
model.params["thickness"].set(value=2.0)     # cm
model.params["α0"].set(value=3000.0)         # moderator response (rise), close to a water moderator
model.params["β0"].set(value=50.0)           # moderator response (decay)
sample = fo.Sample(model, free=["thickness", "strain"])

# the instrument: 9 m flight path, 50 ms period, 10 us bins, a thermal moderator
grid = fo.TimeGrid(flight_path=9.0, frequency=20.0, bin_width=10e-6)
beam = fo.OpenBeam.maxwellian(grid, flux=5e6, area=0.03)
hours = 8

In [2]:
pattern = fo.Pattern.times([0.0, 11.97e-3, 22.33e-3, 35.48e-3], width=200e-6)
exp = fo.Experiment(sample, beam, pattern, time=hours * 3600)
cases = {
    "one pulse 1 us late": fo.Pattern(pattern.starts + np.r_[0, 1e-6, 0, 0], 200e-6),
    "all pulses 1 us late (time zero)": fo.Pattern(pattern.starts + 1e-6, 200e-6),
    "one pulse 1 % stronger, same total": fo.Pattern(pattern.starts, 200e-6, intensity=[1.01, 1 - 0.01 / 3, 1 - 0.01 / 3, 1 - 0.01 / 3]),
    "all pulses 1 % stronger (normalisation)": fo.Pattern(pattern.starts, 200e-6, intensity=1.01),
    "pulses 1 % wider": fo.Pattern(pattern.starts, 202e-6, intensity=200 / 202),
}
import pandas as pd
pd.DataFrame({k: fo.pattern_bias(exp, p) for k, p in cases.items()}).T.round(2)

,thickness,strain
one pulse 1 us late,-0.17,0.42
all pulses 1 us late (time zero),-1.02,1.67
"one pulse 1 % stronger, same total",-0.01,0.05
all pulses 1 % stronger (normalisation),-21.39,-0.23
pulses 1 % wider,-1.04,1.57


The table gives the bias in units of the standard deviation of each
parameter. Pulse times matter for the strain (a digital chopper sets them to
far better than a microsecond); intensities matter for the thickness.

## The cost of freeing what is assumed known

In [3]:
model_r = nbragg.TransmissionModel(xs, response="jorgensen_inv", tof_length=9.0, vary_response=True)
model_r.params["thickness"].set(value=2.0)
model_r.params["α0"].set(value=3000.0)
model_r.params["β0"].set(value=50.0)
rows = {}
for name, free in {"thickness, strain": ["thickness", "strain"],
                   "+ normalisation": ["thickness", "strain", "norm"],
                   "+ moderator response": ["thickness", "strain", "α0", "β0"]}.items():
    s = fo.Sample(model_r, free=free)
    rows[name] = fo.Experiment(s, beam, pattern, time=hours * 3600).precision().set_index("parameter").sigma
pd.DataFrame(rows).T

parameter,norm,strain,thickness,α0,β0
"thickness, strain",NaN,0.000086,0.000432,NaN,NaN
+ normalisation,0.004976,0.000086,0.004622,NaN,NaN
+ moderator response,NaN,0.000264,0.000435,6847.795895,1.358449


Freeing the normalisation costs most of the thickness precision, and
freeing the moderator response costs most of the strain precision: both should
be calibrated in separate measurements.